# Çizim duygu sınıflandırma — Colab

Bu notebook sadece `src/` fonksiyonlarını çağırıp sonuç gösterir. Kod değişikliği yerelde repoda yapılır,
burada `git pull` ile çekilir. Pull'dan sonra değişen kodun yüklenmesi için **Runtime > Restart session** yapıp
hücreleri baştan çalıştır.

Eski keşif ve deneme hücreleri `grad_proj.ipynb`'de (arşiv).

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=True)

import importlib, os, subprocess, sys
REPO = "/content/child-drawing-analysis"
URL = "github.com/nia1da/child-drawing-analysis.git"
# Repo gizliyse Colab Secrets'taki GITHUB_TOKEN kullanılır (sadece bu repo, Contents: Read-only); public ise gerekmez
try:
    yetkili = f"https://x-access-token:{userdata.get('GITHUB_TOKEN')}@{URL}"
except Exception:
    yetkili = f"https://{URL}"

def git(*args):
    # Hata mesajında token görünmesin diye komut yazdırılmaz
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git hatası: " + r.stderr.replace(yetkili, f"https://{URL}"))

if os.path.isdir(f"{REPO}/.git"):
    git("-C", REPO, "pull", "-q", yetkili, "main")
else:
    git("clone", "-q", yetkili, REPO)
    git("-C", REPO, "remote", "set-url", "origin", f"https://{URL}")  # token .git/config'te kalmasın
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

if REPO not in sys.path:
    sys.path.insert(0, REPO)
importlib.invalidate_caches()  # önceki başarısız denemeden kalan "klasör yok" önbelleğini temizler

In [ ]:
import pandas as pd
from src.config import ORTAM, YOLLAR
from src.veri import veri_hazirla, metadata_yukle
from src.egitim import egit, cihaz
from src.degerlendir import degerlendir
print("Ortam:", ORTAM, "| cihaz:", cihaz)

veri_hazirla()  # KIDO zip'ini Drive'dan /content/kido altına açar; zip Drive'da yoksa önce indirip oraya kaydeder
df = metadata_yukle()

# deneyler.csv eski 4 sütunlu düzendeyse 12 sütuna çevirir (orijinal deneyler_eski.csv olarak kalır);
# zaten yeni düzendeyse bir şey yapmaz
from src.deneyler_tasi import tasi
if os.path.exists(f"{YOLLAR['cikti']}/deneyler.csv"):
    tasi()
print(df.bolum.value_counts())

## Test değerlendirmesi

Test seti sadece seçilmiş modellerin son kıyaslaması için kullanılır; model ve hiperparametre seçimi val F1 ile yapılır.

In [ ]:
DENEYLER = [
    "convnext_tiny_448px_lr0.0001_ls0.1_s42",
    "resnet50_448px_lr0.0001_ls0.1_s42",
]
test = {d: degerlendir(df, d) for d in DENEYLER}
pd.DataFrame({d: r["genel"] for d, r in test.items()}).T

## Seed varyansı

convnext_tiny 448px'i farklı seed'lerle tekrar eğit (seed 42 eski kayıtlarda var). Farklar 1–2 puanın
gürültü mü gerçek mi olduğunu gösterir. T4'te her biri ≈ 25 dk.

In [ ]:
for seed in [1, 2]:
    egit(df, "convnext_tiny", boyut=448, seed=seed, not_="seed varyansı")

## Tüm deneyler

In [ ]:
pd.read_csv(f"{YOLLAR['cikti']}/deneyler.csv").sort_values("en_iyi_val_f1", ascending=False)